In [5]:
# Setting up RAG

# Set up the OpenAI client:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

# Load the data and build the search index:
from rag_helper import RAGBase
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

# Create the assistant:
instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
)

In [6]:
# Testing it

# Let's try a question:
assistant.rag("How do I run Ollama locally?")

# This works fine. The search finds relevant FAQ entries about Ollama, and the LLM gives a good answer.

'To run Ollama locally:\n\n1. Install Ollama from https://ollama.com/download for your operating system.\n   - macOS: download and install the `.pkg`\n   - Windows: download and install the `.msi`\n   - Linux: run:\n     ```bash\n     curl -fsSL https://ollama.com/install.sh | sh\n     ```\n\n2. Open a terminal and run:\n   ```bash\n   ollama run llama3\n   ```\n\nThis will download the LLaMA 3 model, start it locally, and open a chat-like interface.\n\n3. To test that the local server is running, you can use:\n   ```bash\n   curl http://localhost:11434\n   ```\n\nIf needed, you can also restart the Ollama server with:\n```bash\n!nohup ollama serve > nohup.out 2>&1 &\n```'

In [7]:
# Now try something slightly different:
assistant.rag("How do I run Olama locally?")

'I don’t see any FAQ entry about running **Ollama** locally.\n\nThe closest relevant guidance is that you can run the course locally if you’re comfortable setting up the needed tools like Python, `uv`, Jupyter, Docker, and others for the module.'

In [8]:
messages = [
    {'role': 'user', 'content': 'I just discovered the course. Can I still join it?'}
]

response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
)

response.output_text

'Absolutely — you should be able to join if enrollment is still open.\n\nIf you want, I can help you figure out:\n- whether the course is still accepting students,\n- how to enroll,\n- or what to say if you need to ask the instructor directly.\n\nIf you’d like, I can also help you write a short message like:\n> Hi, I just discovered the course and wanted to ask if it’s still possible to join. Is enrollment still open?\n\n'

In [9]:
# now instead of sending only a question to the lllm well also give him the function he can use

# defining the tool: 

def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [10]:
search('how do i run ollama')

[{'id': '1d0b969028',
  'course': 'llm-zoomcamp',
  'section': 'Module 1: RAG',
  'question': 'Ollama: How to install Ollama?',
  'answer': 'First, install Ollama by visiting [https://ollama.com/download](https://ollama.com/download) and choosing your operating system:\n\n- **macOS**: Download the `.pkg` and install it.\n- **Windows**: Download the `.msi` and install it.\n- **Linux**: Run the following command in the terminal:\n\n  ```bash\n  curl -fsSL https://ollama.com/install.sh | sh\n  ```\n\nOnce installed, open a terminal and type:\n\n```bash\nollama run llama3\n```\n\nThis command will:\n\n- Download the LLaMA 3 model (~4GB).\n- Start the model locally.\n- Open a chat-like interface where you can type questions.\n\nTo test the Ollama local server, run the following command:\n\n```bash\ncurl http://localhost:11434\n```\n\nYou should receive a response similar to:\n\n```json\n{"models": [...]}  \n```\n\nThen, install the Python client with:\n\n```bash\npip install ollama\n```\n\n

In [11]:
# llms are language agnostic. we just making an http call to some endpoint, and we need 
# a language agnostic way of defining a function:

search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [12]:
response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
    tools=[search_tool]
)


In [13]:
len(response.output)

1

In [14]:
call = response.output[0]

In [15]:
# llm already changed the question (this is not the question in the way we)
call

ResponseFunctionToolCall(arguments='{"query":"join course late discovered can I still join"}', call_id='call_dgqezusRg9HsuBpE4q0mwM24', name='search', type='function_call', id='fc_058ba93e10132140006a39c1665f68819eaaa71b8119337935', namespace=None, status='completed')

In [16]:
# this was the tool he is using
call.name

'search'

In [17]:
# now we need to parse these arguments

import json

# this is the query he is using 
args = json.loads(call.arguments)

In [18]:
results = search(**args)
results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '9f689c185f',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I missed the first homework - can I still get a certificate?',
  'answer': 'Yes, you need to pass the Capstone project to get the certificate. Homework is not mandatory, though it is recommended for reinforcing concepts, and the points awarded count towards your rank on the leaderboard.'},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\n

In [19]:
# what happenend?
# history


# 1. make a call to te LLM <-- we pay
# 2. LLM decided to invoke search('params')
# 3. we invoke the search , we have the results 
# 4. send the results back to the llm (another call) <-- we pay again
# 5. llm processes the results 
# 6. llm gives the answer

In [20]:
# -> when we turn rag into agentic rag, we need to do more!

# turn everything into json what well send to the llm

result_json = json.dumps(results, indent=2)
print(result_json)

[
  {
    "id": "74eb249bbf",
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "I just discovered the course. Can I still join?",
    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\u2019re still accepting submissions."
  },
  {
    "id": "9f689c185f",
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "I missed the first homework - can I still get a certificate?",
    "answer": "Yes, you need to pass the Capstone project to get the certificate. Homework is not mandatory, though it is recommended for reinforcing concepts, and the points awarded count towards your rank on the leaderboard."
  },
  {
    "id": "69d122f12e",
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "Certificate: Can I follow the course in a self-paced mode and get a certificate?",
    "answer": "No, you can only get a certifi

In [21]:
# we received a functionToolCall response, a decision of an llm to use a tool
# basically: he told us to invoke the function, we used the function,  
# there i a related call id (bc we need to know for which call it was needed)

function_call_output = {
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
}

In [22]:
# llms are stateless! every time they need the entire previous history!

# append decision
messages.append(call)

# append output
messages.append(function_call_output)

print(messages)

[{'role': 'user', 'content': 'I just discovered the course. Can I still join it?'}, ResponseFunctionToolCall(arguments='{"query":"join course late discovered can I still join"}', call_id='call_dgqezusRg9HsuBpE4q0mwM24', name='search', type='function_call', id='fc_058ba93e10132140006a39c1665f68819eaaa71b8119337935', namespace=None, status='completed'), {'type': 'function_call_output', 'call_id': 'call_dgqezusRg9HsuBpE4q0mwM24', 'output': '[\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."\n  },\n  {\n    "id": "9f689c185f",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I missed the first homework - can I still get a certificate?",\n    "answer": "Yes, you need to

In [23]:
# send results back to the llm to process the results 

response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
    tools=[search_tool]
)

In [24]:
# what could happen now is: either it actually has an answer for us, or it decides to make another searhch because it doesnt have enough information


print(response.output_text)

Yes — you can still join anytime.

If you want a certificate, make sure to submit your project while submissions are still open.


In [25]:
# summary: the path was a bit longer but now the agent is in control and can decide whats happening with the user 
# query and what to do with the too: now the llm is thedriver!

# that is what makes the call to the llm agentic!

# agent has a certain task: (in this case to help the user student)
# agent has memory
# agent has tools (to actually carry out the task)



In [26]:
# how much we payed for this? 

usage = response.usage
print(usage.input_tokens, usage.output_tokens) 

prompt = """ 
i use gpt 5-4 min
(814,35)
this is input output tokens
write a python function to calculate the price and also give the price
"""

810 30


In [27]:
def calculate_gpt54_mini_cost(input_tokens: int, output_tokens: int):
    INPUT_PRICE_PER_MILLION = 0.25   # USD
    OUTPUT_PRICE_PER_MILLION = 2.00  # USD

    input_cost = input_tokens / 1_000_000 * INPUT_PRICE_PER_MILLION
    output_cost = output_tokens / 1_000_000 * OUTPUT_PRICE_PER_MILLION
    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
    }


cost = calculate_gpt54_mini_cost(814, 35)
print(cost)

{'input_cost': 0.0002035, 'output_cost': 7e-05, 'total_cost': 0.0002735}


In [28]:
# THE AGENTIC LOOP

# What if the LLM decides to make multiple calls?

# what happenend?
# history


# 1. make a call to te LLM <-- we pay
# 2. LLM decided to invoke search('params')
# 3. we invoke the search , we have the results 
# 4. send the results back to the llm (another call) <-- we pay again
# 5. llm processes the results 
# 6. LLM decides to make another tool call
# 7. send one more api request 
# 8. process and gives the answer 
# ...

# for that, we need: 
# 1. developer prompt: the role and behavior
# ...

# -> agent(instructions, memory, tools)

# we exit the loop when there are no more tool calls 


In [29]:
# setup

def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)
    # if call.name == "some other fuction"

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

question = "I just discovered the course. Can I join it?"

messages = [
    {'role': 'developer', 'content': instructions},
    {'role': 'user', 'content': question}
]



In [30]:
response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
    tools=[search_tool]
)

response.output_text

''

In [31]:
response.output

[ResponseFunctionToolCall(arguments='{"query":"join course discovered course can I join enrollment late registration access"}', call_id='call_RtQToD8YqPOsGDf3OZILJGKT', name='search', type='function_call', id='fc_0746ade8e9c49cfa006a39c168a148819c97a07be0b864718c', namespace=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"query":"course FAQ join after start discovered the course enrollment open"}', call_id='call_q3UDPNRUtBO5Z862Sqf6TFhp', name='search', type='function_call', id='fc_0746ade8e9c49cfa006a39c168a158819cb42a9b51b9c2cfaa', namespace=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"query":"can I still join the course FAQ late enrollment"}', call_id='call_ctsizXGAtOK0v7TGMtig4ZZ2', name='search', type='function_call', id='fc_0746ade8e9c49cfa006a39c168a160819ca60ee5855ab64b1e', namespace=None, status='completed')]

In [32]:
response.output[1]

ResponseFunctionToolCall(arguments='{"query":"course FAQ join after start discovered the course enrollment open"}', call_id='call_q3UDPNRUtBO5Z862Sqf6TFhp', name='search', type='function_call', id='fc_0746ade8e9c49cfa006a39c168a158819cb42a9b51b9c2cfaa', namespace=None, status='completed')

In [33]:
# add to history (multiple items in the output now)
messages.extend(response.output)

for item in response.output:

    if item.type == "function_call":
        # print("function_call: ", item.name, item.arguments)
        # call_output = make_call(item)
        # messages.append(call_output)

        print("function_call:", item.name, item.arguments)
        # invoke this helper 
        call_output = make_call(item)
        # add to history
        messages.append(call_output)

    elif item.type == "message":
        print("ASSISTANT: ")
        print(item.content[0].text)
        

function_call: search {"query":"join course discovered course can I join enrollment late registration access"}
function_call: search {"query":"course FAQ join after start discovered the course enrollment open"}
function_call: search {"query":"can I still join the course FAQ late enrollment"}


In [34]:
messages

[{'role': 'developer',
  'content': "You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches.\n\nTry to expand your search by using new keywords\nbased on the results you get from the search.\n\nAt the end, ask if there are other areas that the user wants to explore."},
 {'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseFunctionToolCall(arguments='{"query":"join course discovered course can I join enrollment late registration access"}', call_id='call_RtQToD8YqPOsGDf3OZILJGKT', name='search', type='function_call', id='fc_0746ade8e9c49cfa006a39c168a148819c97a07be0b864718c', namespace=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"query":"course FAQ join after start discovered the course enrollment open"}

In [35]:
# FOLLOW UP QUESTIONS 

In [36]:
# reset messages 
messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

iterations = 1

while True:
    print(f"iteration {iterations}...")
    has_function_calls = False

    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=messages,
        tools=[search_tool],
    )

    # add to history (multiple items in the output now)
    messages.extend(response.output)

    for item in response.output:

        if item.type == "function_call":
            # print("function_call: ", item.name, item.arguments)
            # call_output = make_call(item)
            # messages.append(call_output)

            print("function_call:", item.name, item.arguments)
            # invoke this helper 
            call_output = make_call(item)
            # add to history
            messages.append(call_output)
            has_function_calls = True

        elif item.type == "message":
            print("ASSISTANT: ")
            print(item.content[0].text)

    iterations = iterations + 1
    if has_function_calls == False:
        break

        



iteration 1...


function_call: search {"query":"join the course late enrollment discovered the course can I join"}
function_call: search {"query":"course enrollment registration late join discovered course"}
iteration 2...
ASSISTANT: 
Yes — you can still join the course.

You can start learning now, even if you just discovered it. The videos and materials are available, and you can work through the lessons at your own pace. One important note: if you want a certificate, you need to submit your project while submissions are still open.

If you want, I can also help you with how to start the course or explain the weekly workflow.


In [37]:
# DIFFERENT INSTRUCTIONS 


instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results and then perform more searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

question = "I just discovered the course. Can I join it?"

messages = [
    {'role': 'developer', 'content': instructions},
    {'role': 'user', 'content': question}
]

iterations = 1

while True:
    print(f"iteration {iterations}...")
    has_function_calls = False

    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=messages,
        tools=[search_tool],
    )

    # add to history (multiple items in the output now)
    messages.extend(response.output)

    for item in response.output:

        if item.type == "function_call":
            # print("function_call: ", item.name, item.arguments)
            # call_output = make_call(item)
            # messages.append(call_output)

            print("function_call:", item.name, item.arguments)
            # invoke this helper 
            call_output = make_call(item)
            # add to history
            messages.append(call_output)
            has_function_calls = True

        elif item.type == "message":
            print("ASSISTANT: ")
            print(item.content[0].text)

    iterations = iterations + 1
    if has_function_calls == False:
        break 


iteration 1...
function_call: search {"query":"join course discovered course can I join enrollment register late join FAQ"}
iteration 2...
function_call: search {"query":"certificate project submitting while accepting submissions live cohort self-paced join course FAQ"}
iteration 3...
ASSISTANT: 
Yes — you can still join the course.

A couple of important notes:
- You can start learning and submit homework while the submission form is open.
- If you want a certificate, you need to submit your project before submissions close and be part of the live cohort process.

If you'd like, I can also help you figure out how to start the course or explain the certificate requirements. Any other areas you want to explore?


In [38]:
# put that into a function 

def agent_loop(instructions, question, model="gpt-5.4-mini") -> str:



    messages = [
        {'role': 'developer', 'content': instructions},
        {'role': 'user', 'content': question}
    ]

    iterations = 1

    while True:
        print(f"iteration {iterations}...")
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool],
        )

        # add to history (multiple items in the output now)
        messages.extend(response.output)

        for item in response.output:

            if item.type == "function_call":
                # print("function_call: ", item.name, item.arguments)
                # call_output = make_call(item)
                # messages.append(call_output)

                print("function_call:", item.name, item.arguments)
                # invoke this helper 
                call_output = make_call(item)
                # add to history
                messages.append(call_output)
                has_function_calls = True

            elif item.type == "message":
                print("ASSISTANT: ")
                last_answer = item.content[0].text
                print(last_answer)

        iterations = iterations + 1
        if has_function_calls == False:
            break 

    return last_answer

        

In [39]:

instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results and then perform more searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

question = "I just discovered the course. Can I join it?"

result = agent_loop(
    instructions=instructions,
    question=question
)

iteration 1...
function_call: search {"query":"join course discovered late enroll can I join course FAQ"}
iteration 2...
function_call: search {"query":"certificate project while accepting submissions peer-review live cohort self-paced FAQ"}
iteration 3...
ASSISTANT: 
Yes — you can still join the course.

If you want a certificate, though, you need to submit your project while submissions are still open, and the course’s peer-review process has to be active. In other words, you can follow along anytime, but certificate eligibility depends on being able to complete the live cohort requirements.

If you want, I can also explain:
- how to start the course now,
- whether you can do it self-paced,
- or how the certificate/peer-review process works.


In [40]:
result

'Yes — you can still join the course.\n\nIf you want a certificate, though, you need to submit your project while submissions are still open, and the course’s peer-review process has to be active. In other words, you can follow along anytime, but certificate eligibility depends on being able to complete the live cohort requirements.\n\nIf you want, I can also explain:\n- how to start the course now,\n- whether you can do it self-paced,\n- or how the certificate/peer-review process works.'

In [41]:
question = "what's queen gambit?"

result = agent_loop(
    instructions=instructions,
    question=question
)

iteration 1...
function_call: search {"query":"queen gambit"}
iteration 2...
function_call: search {"query":"queen's gambit chess opening"}
iteration 3...
function_call: search {"query":"queen gambit chess opening definition"}
iteration 4...
ASSISTANT: 
“Queen’s Gambit” is a chess opening that starts with:

1. d4 d5  
2. c4

White offers the c-pawn to try to draw Black’s d-pawn away from the center and gain positional control.

A key idea:
- White is not really “giving up” the pawn permanently.
- The opening is about controlling the center and getting active development.

Common responses from Black:
- **Accepted**: 2... dxc4
- **Declined**: 2... e6 or 2... c6

If you meant the Netflix show, “The Queen’s Gambit” is also the name of a series inspired by chess.

If you want, I can also explain:
- the main ideas behind the Queen’s Gambit,
- how it differs from the King’s Gambit,
- or show a few simple opening lines.


In [42]:
# adjust insstructions again 


instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results and then perform more searches.

The question has to be about thye course or its logistics, offtopic questions shoudnt be answered.
If the serach returns nothing, it's likely an off-topic question. 
If you can't answer the question using FAQ, don't do it yourself. 
Only use the facts from the FAQ database. 

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

question = "what's queen gambit?"

result = agent_loop(
    instructions=instructions,
    question=question
)



iteration 1...


function_call: search {"query":"queen gambit"}
iteration 2...
function_call: search {"query":"gambit chess opening queen's gambit"}
iteration 3...
function_call: search {"query":"queen's gambit chess opening"}
iteration 4...
ASSISTANT: 
I couldn’t find anything in the course FAQ about “queen’s gambit” as a course topic, so it looks like this is off-topic for the course.

If you meant a course-specific term or logistics question, feel free to ask that instead. Is there another area you want to explore?


In [43]:
# NOW WE WILL USE A FRAMEWORK INSTEAD OF DOING EVERYTHING MANUALLY

In [44]:
!uv add toyaikit

Resolved 129 packages in 79ms
Checked 125 packages in 335ms


In [45]:
# we need imports 

from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [46]:
agent_tools = Tools()
# actual tool / description of the tool
agent_tools.add_tool(search, search_tool)

# also annoying to write this description everytime by hand or with chatgpt
# typically we add docstring and typehints instead to the tool function 

In [47]:
# better: 
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [48]:
# now instead of passing the description we do: 

agent_tools = Tools()
agent_tools.add_tool(search)

agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [49]:
# he mentioned the frameworks briedflyu: 

# openai agents jdk 
# pedantic ai 
# google adk
# langchain? 
# krueger?? 
# i didnt understand everything what he said
# 
#  


In [50]:
# we define the actual agent 

# toyaikit is doing what we did so far, but a little bit more interactive

chat_interface = IPythonChatInterface()  # this displays the output
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

In [51]:
# we want to do this a little bit more interactive: 

# everytime something happens, for example tool call or a message is display it will invoke the callback to display things.


result = runner.loop(
    prompt="How do I run Olama locally?",
    callback=callback,
)

-> Response received


-> Response received


-> Response received


In [52]:
result.cost

CostInfo(input_cost=Decimal('0.0027735'), output_cost=Decimal('0.0013545'), total_cost=Decimal('0.0041280'))

In [53]:
result.all_messages

[EasyInputMessage(content="You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches. First perform search, analyze the results and then perform more searches.\n\nThe question has to be about thye course or its logistics, offtopic questions shoudnt be answered.\nIf the serach returns nothing, it's likely an off-topic question. \nIf you can't answer the question using FAQ, don't do it yourself. \nOnly use the facts from the FAQ database. \n\nTry to expand your search by using new keywords\nbased on the results you get from the search.\n\nAt the end, ask if there are other areas that the user wants to explore.", role='developer', phase=None, type=None),
 EasyInputMessage(content='How do I run Olama locally?', role='user', phase=None, type=None),
 ResponseFunct

In [54]:
# if we want to continue the conversation?

result2 = runner.loop(
    previous_messages=result.all_messages,
    prompt="How do I run a different model?",
    callback=callback,
)

-> Response received


-> Response received


In [55]:
# Interactive chat
# runner.run();

In [56]:
# OTHER FRAMEWORKS 

"""
His favorite is PydanticAI. 

he then explained the other ones 

"""


# agent

"""
- has instructions 
- has tools 
- has memory

-> this is how we turn the rigig rag into something more dynamic.

-> we don't always need an agent. maybe sometimes we only need 
smart parsing of a document into another, or we can use only rigid rag

-> we should try first without usin an agent, because 
its takes longer, it needs to explore things, we need frameworks, 
we pay more, and also agents can go wild and do toolcall after toolcall etc 


then we hav to monitor costs and whats happening , so if we dont really need 
this complexity for our project then we try to avoid it

but if we tried all other things and can be confident that we need and 
we can justify the additional complexity and costs
"""

"\n- has instructions \n- has tools \n- has memory\n\n-> this is how we turn the rigig rag into something more dynamic.\n\n-> we don't always need an agent. maybe sometimes we only need \nsmart parsing of a document into another, or we can use only rigid rag\n\n-> we should try first without usin an agent, because \nits takes longer, it needs to explore things, we need frameworks, \nwe pay more, and also agents can go wild and do toolcall after toolcall etc \n\n\nthen we hav to monitor costs and whats happening , so if we dont really need \nthis complexity for our project then we try to avoid it\n\nbut if we tried all other things and can be confident that we need and \nwe can justify the additional complexity and costs\n"

In [57]:
# homework

In [58]:
"""
Setup
Prepare your environment the same way as in the module's Environment lesson.

This homework needs one extra library: gitsource, which downloads files from a GitHub repository.

Install it:

uv add gitsource
For the LLM, we recommend OpenAI with gpt-5.4-mini, but you can use any model and provider you like - just adapt the client and the usage fields accordingly.
"""

# done from terminal

"\nSetup\nPrepare your environment the same way as in the module's Environment lesson.\n\nThis homework needs one extra library: gitsource, which downloads files from a GitHub repository.\n\nInstall it:\n\nuv add gitsource\nFor the LLM, we recommend OpenAI with gpt-5.4-mini, but you can use any model and provider you like - just adapt the client and the usage fields accordingly.\n"

In [59]:
"""
Preparation
First, we will pull the lesson pages straight from the course repository. We will use the commit 8c1834d to make sure everyone works with the exact same data.

We will use gitsource for that:

from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()
GithubRepositoryDataReader downloads the entire repository and goes over all the files in it. Because we specify allowed_extensions={"md"}, it only checks the markdown files.

We also pass a filename_filter so we don't grab every markdown file in the repo, like the top-level README. The lesson pages all live under a module's lessons/ folder, so filtering on /lessons/ keeps just those.

Each file has a parse() method that returns a dictionary with its filename and content:

documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)
"""

'\nPreparation\nFirst, we will pull the lesson pages straight from the course repository. We will use the commit 8c1834d to make sure everyone works with the exact same data.\n\nWe will use gitsource for that:\n\nfrom gitsource import GithubRepositoryDataReader\n\nreader = GithubRepositoryDataReader(\n    repo_owner="DataTalksClub",\n    repo_name="llm-zoomcamp",\n    commit_id="8c1834d",\n    allowed_extensions={"md"},\n    filename_filter=lambda path: "/lessons/" in path,\n)\n\nfiles = reader.read()\nGithubRepositoryDataReader downloads the entire repository and goes over all the files in it. Because we specify allowed_extensions={"md"}, it only checks the markdown files.\n\nWe also pass a filename_filter so we don\'t grab every markdown file in the repo, like the top-level README. The lesson pages all live under a module\'s lessons/ folder, so filtering on /lessons/ keeps just those.\n\nEach file has a parse() method that returns a dictionary with its filename and content:\n\ndocume

In [60]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [61]:
files [0]

RawRepositoryFile(filename='01-agentic-rag/lessons/01-intro.md', content='# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are 

In [62]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

In [63]:
# Q1. How many lesson pages
# How many lesson pages are in the dataset?

# 24
# 72
# 240
# 720

len(documents)
# 72

72

In [64]:
# Q2. Indexing and searching
# Index the documents with minsearch - make content a text field and filename a keyword field. Then search with this query:

# How does the agentic loop keep calling the model until it stops?

# What's the filename of the first result?

# 01-agentic-rag/lessons/03-rag.md
# 01-agentic-rag/lessons/14-agentic-loop.md
# 04-evaluation/lessons/13-llm-as-judge.md
# 06-best-practices/lessons/02-hybrid-search.md

from rag_helper import RAGBase
# from ingest import load_faq_data, build_index


# ai client 
from dotenv import load_dotenv
load_dotenv() 

from openai import OpenAI
openai_client = OpenAI()

# ingest 

import requests
from minsearch import Index



def build_index(documents):
    index = Index(
        text_fields=["content"],
        keyword_fields=["filename"]
    )
    index.fit(documents)
    return index

# documents = load_faq_data()
index = build_index(documents)




In [65]:
# boost_dict = {"question": 3.0, "section": 0.5}
query = "How does the agentic loop keep calling the model until it stops?"
filter_dict = {"course": "llm-zoomcamp"}

search_result = index.search(
    query,
    # num_results=num_results,
    # boost_dict=boost_dict,
    # filter_dict=filter_dict
)

search_result

# filename': '01-agentic-rag/lessons/14-agentic-loop.md'

[{'content': '# The Agentic Loop\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=ePlQUcTPPjw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we did function calling by hand. We sent a\nmessage and got back a function call. We ran it, sent the result back,\nand got the answer.\n\nThat works for one function call. It breaks down when the model wants\nto search several times, or when the first search misses the answer.\nWe don\'t know in advance how many calls the model will want. So we\nneed a loop that keeps calling the model and running tools until it\'s\ndone. An agent is exactly that.\n\n## Anatomy of an agent\n\nWith the LLM in the driver\'s seat, we have an agent. It\'s an AI\nassistant whose goal is to help the user.\n\nAn agent has three parts:\n\n- Instructions, the role and behavior we want. We pass this as the\n  `developer` message. The better the instructions, the better the\n  agent helps.\n- Tools, the functions the agent can call to carry

In [66]:
# q3 

# will contain the rag function: 


# they need documents / index global variable, but i can not create it here in this way: 
# documents = load_faq_data()
# index = build_index(documents)

# -> because this is not flexible and i cant really go and replace the load and index fct with 
# something else 

# INSTEAD: WE WILL CREATE A CLASS
# then i can put into search any index that i want 

INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

USER_PROMPT_TEMPLATE = """
QUESTION: {question}

CONTEXT:
{context}
""".strip()

class RAGBase:

    def __init__(
        self,
        index,
        llm_client,
        instructions=INSTRUCTIONS,
        prompt_template=USER_PROMPT_TEMPLATE,
        course="llm-zoomcamp",
        model="gpt-5.4-mini"
    ):
        self.index = index
        self.llm_client = llm_client
        self.instructions = instructions
        self.course = course
        self.prompt_template = prompt_template
        self.model = model

    # search 

    def search(self, query: str) -> dict[str, str]:
        """
        Search the course readme documents for entries matching the given query.
        """

        # filter_dict = {"course": self.course}

        return self.index.search(
            query,
            # num_results=num_results,
            # boost_dict=boost_dict,
            # filter_dict=filter_dict
        )

    def build_context(self, search_results):
        lines = []

        for doc in search_results:
            # lines.append(doc["section"])
            lines.append("content: " + doc["content"])
            lines.append("filename: " + doc["filename"])
            lines.append("")

        return "\n".join(lines).strip()

    def build_prompt(self, query, search_results):
        context = self.build_context(search_results)
        return self.prompt_template.format(
            question=query, context=context
        )

    # if i wanted to use another provider model this would the function i'd overwrite:
    # i'd create a class classed OllamaRAG(RAGBase):
        # def some other llm function: 
        #     ... 
    def llm(self, prompt):
        input_messages = [
            {"role": "developer", "content": self.instructions},
            {"role": "user", "content": prompt}
        ]

        response = self.llm_client.responses.create(
            model=self.model,
            input=input_messages
        )

        return response.output_text, response.usage

    def rag(self, query):
        search_results = self.search(query)
        prompt = self.build_prompt(query, search_results)
        answer = self.llm(prompt)
        return answer


In [67]:

assistant = RAGBase(
    index=index,
    llm_client=openai_client
) # we dont need to overwrite anything here, but if we wanted to, we could

response_q3 = assistant.rag('How does the agentic loop keep calling the model until it stops?')

In [68]:
response_q3

('The loop keeps calling the model until the response comes back **without any function calls**.\n\nIn the code, it works like this:\n\n- call the model\n- check each item in `response.output`\n- if there is a `function_call`, run the tool and append the tool result to `messages`\n- set `has_function_calls = True`\n- repeat the loop\n- if `has_function_calls == False`, `break`\n\nSo the stopping condition is simply: **no tool calls this turn means the agent is done**.',
 ResponseUsage(input_tokens=10339, input_tokens_details=InputTokensDetails(cached_tokens=9984), output_tokens=112, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=10451))

In [69]:
response_q3_usage = response_q3[1]
response_q3_usage

# input tokes: 10339

# closes from the answers : 7000

# 700
# 7000
# 70000
# 700000

ResponseUsage(input_tokens=10339, input_tokens_details=InputTokensDetails(cached_tokens=9984), output_tokens=112, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=10451)

In [70]:
# q4: 

from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)
len(chunks)

# 295

295

In [71]:
# q5 :


index = build_index(chunks)



assistant = RAGBase(
    index=index,
    llm_client=openai_client
) # we dont need to overwrite anything here, but if we wanted to, we could

response_q5 = assistant.rag('How does the agentic loop keep calling the model until it stops?')


response_q5_usage = response_q5[1]
response_q5_usage

# input tokes: 4787

# closes from the answers : 3 times fewer 

# about the same
# 3× fewer
# 10× fewer
# 30× fewer


ResponseUsage(input_tokens=4787, input_tokens_details=InputTokensDetails(cached_tokens=4352), output_tokens=86, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=4873)

In [72]:
# q6: 
# !uv add toyaikit


instructions_for_q6 = """
You're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering.

"""

# we need imports 

from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback


agent_tools = Tools()
agent_tools.add_tool(assistant.search)

agent_tools.get_tools()


[{'type': 'function',
  'name': 'search',
  'description': 'Search the course readme documents for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [ ]:


# we define the actual agent 

# toyaikit is doing what we did so far, but a little bit more interactive

chat_interface = IPythonChatInterface()  # this displays the output
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions_for_q6,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

# we want to do this a little bit more interactive: 

# everytime something happens, for example tool call or a message is display it will invoke the callback to display things.


result = runner.loop(
    prompt="How does the agentic loop work, and how is it different from plain RAG?",
    callback=callback,
)


# it ran 3 times. closest option among this is 4: 
# 0
# 4
# 10
# 20




-> Response received


-> Response received
